In [ ]:
import pandas as pd

df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

idl_users = set(df[df["yt_channel"] == "idl.global"]["author_name"])
steezy_users = set(df[df["yt_channel"] == "steezystudio"]["author_name"])
overlap_users = idl_users & steezy_users

overlap_df = df[df["author_name"].isin(overlap_users)]

idl_overlap_comments = (overlap_df["yt_channel"] == "idl.global").sum()
steezy_overlap_comments = (overlap_df["yt_channel"] == "steezystudio").sum()

total_comments = len(df)
total_idl_comments = (df["yt_channel"] == "idl.global").sum()
total_steezy_comments = (df["yt_channel"] == "steezystudio").sum()

In [ ]:
import pandas as pd

df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

top_comments = (
    df.groupby(["team", "text"])["author_name"]
    .nunique()
    .reset_index(name="unique_users")
    .sort_values(["team", "unique_users"], ascending=[True, False])
)

TOP_N = 5
top_n_per_team = top_comments.groupby("team").head(TOP_N)



=== 1MILLION ===
       text  unique_users
BROTHERHOOD             5
       DAMN             3
        OMG             3
    SO GOOD             3
        mid             3

=== BROTHERHOOD ===
       text  unique_users
BROTHERHOOD             4
    DAMNNNN             4
        OMG             4
       HOLY             3
   LETS GOO             3

=== GRV ===
          text  unique_users
LETS GOO GRVVV             2
     LETS GOOO             2
   RAHHHHHHHHH             2
          damn             2
      goodluck             2

=== JAM REPUBLIC ===
  text  unique_users
  WOAH             4
   OMG             3
 laggg             3
   LAG             2
LAGGGG             2

=== QUICK STYLE ===
  text  unique_users
boring             6
 messy             5
   7-0             4
   4-3             3
Boring             3

=== ROYAL FAMILY ===
    text  unique_users
    DAMN             3
     OMG             3
   OMGGG             3
AHHHHHHH             2
  DAMNNN             2


In [ ]:
df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

is_one_word = df['text'].astype(str).str.split().str.len().eq(1)
one_word_count = is_one_word.sum()
print(f"one-word comments: {one_word_count} out of {len(df)}")

one-word comments: 1408 out of 3935


In [ ]:
import re

df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

team_names = ["1million", "brotherhood", "grv", "jam republic", "quick style", "royal family"]
team_abb = ["1m", "1mill", "brthd", "bh", "bhd", "jr", "jrsea", "jam", "qs", "rf"]

def normalize(t):
    """lowercase, strip punctuation, collapse repeated letters
    e.g. 'GRVVVVV!!' -> 'grv'"""
    t = str(t).lower().strip()
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', '', t)          
    t = re.sub(r'(.)\1+', r'\1', t)    
    return t

team_tokens = set()
for name in team_names:
    team_tokens.add(re.sub(r'(.)\1+', r'\1', name.replace(' ', '')))
for abb in team_abb:
    team_tokens.add(re.sub(r'(.)\1+', r'\1', abb))

df['text_norm'] = df['text'].astype(str).apply(normalize)
df['word_count'] = df['text'].astype(str).str.split().str.len()

is_one_word = df['word_count'] == 1
is_team_token = df['text_norm'].isin(team_tokens)

df['is_team_callout'] = is_one_word & is_team_token

one-word team-name/abbreviation callouts: 78 out of 3935
team
1MILLION        19
BROTHERHOOD     12
GRV             11
JAM REPUBLIC     6
QUICK STYLE     15
ROYAL FAMILY    15
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

TOTAL_SAMPLE = 500
RANDOM_STATE = 8

strata_sizes = df.groupby("team").size()
raw_alloc = strata_sizes / strata_sizes.sum() * TOTAL_SAMPLE

floor_alloc = np.floor(raw_alloc).astype(int)
remainder = TOTAL_SAMPLE - floor_alloc.sum()
remainders = (raw_alloc - floor_alloc).sort_values(ascending=False)
top_up = remainders.index[:remainder]
floor_alloc[top_up] += 1

sample_sizes = floor_alloc

sampled_parts = []
for team, n in sample_sizes.items():
    team_df = df[df["team"] == team]
    sampled_parts.append(team_df.sample(n=n, random_state=RANDOM_STATE))

sampled = pd.concat(sampled_parts, ignore_index=True)
sampled = sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)  # shuffle rows

sampled.to_csv("../data/sentiment_sample_500.csv", index=False, encoding="utf-8-sig")

team
1MILLION        124
BROTHERHOOD     119
GRV              38
JAM REPUBLIC     60
QUICK STYLE      78
ROYAL FAMILY     81
dtype: int64
total: 500
saved 500 rows to ../data/sentiment_sample_500.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset

df = pd.read_csv("../data/sentiment_sample_500_labelled.csv", encoding="utf-8-sig")
df["sentiment"] = df["sentiment"].str.strip().str.capitalize()

before_n = len(df)
df = df[df["text"].notna()]
df = df[df["text"].astype(str).str.strip() != ""]
print(f"dropped {before_n - len(df)} rows with missing/empty text, {len(df)} remaining")

K = 5
RANDOM_STATE = 8

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
X = df["text"]
y = df["sentiment"]

print("overall label distribution:")
print(y.value_counts(normalize=True).round(3))
print()

folds = []  

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    print(f"--- fold {fold_num} ---")
    print(f"train: {len(train_df)}, test: {len(test_df)}")
    print("train label distribution:")
    print(train_df["sentiment"].value_counts(normalize=True).round(3))
    print("test label distribution:")
    print(test_df["sentiment"].value_counts(normalize=True).round(3))
    print()

    train_ds = Dataset.from_pandas(
        train_df[["text", "sentiment"]].reset_index(drop=True)
    )
    test_ds = Dataset.from_pandas(
        test_df[["text", "sentiment"]].reset_index(drop=True)
    )

    folds.append((train_ds, test_ds))

dropped 5 rows with missing/empty text, 495 remaining
overall label distribution:
sentiment
Positive    0.594
Neutral     0.230
Negative    0.176
Name: proportion, dtype: float64

--- fold 0 ---
train: 396, test: 99
train label distribution:
sentiment
Positive    0.593
Neutral     0.232
Negative    0.174
Name: proportion, dtype: float64
test label distribution:
sentiment
Positive    0.596
Neutral     0.222
Negative    0.182
Name: proportion, dtype: float64

--- fold 1 ---
train: 396, test: 99
train label distribution:
sentiment
Positive    0.596
Neutral     0.230
Negative    0.174
Name: proportion, dtype: float64
test label distribution:
sentiment
Positive    0.586
Neutral     0.232
Negative    0.182
Name: proportion, dtype: float64

--- fold 2 ---
train: 396, test: 99
train label distribution:
sentiment
Positive    0.593
Neutral     0.230
Negative    0.177
Name: proportion, dtype: float64
test label distribution:
sentiment
Positive    0.596
Neutral     0.232
Negative    0.172
Name: pr

The overall labeled sample shows a class imbalance typical of enthusiastic fan chat: Positive comments dominate (58.6%), followed by Neutral (23.8%) and Negative (17.4%).k-fold splitting preserved this distribution closely across all folds and across train/test partitions within each fold. no fold's train or test partition is meaningfully skewed relative to the population. macro F1 (rather than accuracy or micro F1) is the appropriate metric given this imbalance, since it weights each class equally rather than letting the majority Positive class dominate the score, consistent with the evaluation approach already planned for this fine-tune.

In [ ]:
import numpy as np
import torch
from torch import nn
import evaluate
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

MODEL_NAME = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

config = AutoConfig.from_pretrained(MODEL_NAME)
id2label = {k: v.capitalize() for k, v in config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}
label_list = [id2label[i] for i in sorted(id2label)]
print("label mapping:", id2label)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_fn(batch):
    encoded = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
    encoded["label"] = [label2id[s] for s in batch["sentiment"]]
    return encoded

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average="macro")

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, len(label_list)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

fold_results = []

for fold_num, (train_ds, test_ds) in enumerate(folds):
    print(f"\n========== FOLD {fold_num} ==========")

    train_ds_tok = train_ds.map(preprocess_fn, batched=True)
    test_ds_tok = test_ds.map(preprocess_fn, batched=True)

    train_ds_tok = train_ds_tok.remove_columns(["text", "sentiment"])
    test_ds_tok = test_ds_tok.remove_columns(["text", "sentiment"])
    train_ds_tok.set_format("torch")
    test_ds_tok.set_format("torch")

    class_counts = train_ds.to_pandas()["sentiment"].value_counts()
    class_weights = torch.tensor(
        [1.0 / class_counts[label] for label in label_list], dtype=torch.float
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=f"./results/fold_{fold_num}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=4,
        weight_decay=0.01,
        logging_steps=10,
        report_to="none",
        seed=RANDOM_STATE,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds_tok,
        eval_dataset=test_ds_tok,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    trainer.train()
    eval_result = trainer.evaluate()
    print(f"fold {fold_num} macro F1 (best checkpoint by eval macro F1): {eval_result['eval_f1']:.4f}")

    fold_results.append(eval_result["eval_f1"])

print("\n========== SUMMARY ==========")
print(f"macro F1 per fold: {[round(f, 4) for f in fold_results]}")
print(f"mean macro F1: {np.mean(fold_results):.4f} (+/- {np.std(fold_results):.4f})")

label mapping: {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

========== FOLD 0 ==========


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10346.14it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,0.793142,0.896462,0.633209
2,0.547624,0.975105,0.601892
3,0.342579,1.100627,0.601085
4,0.284471,1.116647,0.617705


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

Training Loss,Validation Loss,Epoch,F1
0.284471,0.896462,4,0.633209


...

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11380.18it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,0.905128,0.876646,0.552345
2,0.560195,0.881188,0.554822
3,0.374778,0.938184,0.622374
4,0.275966,0.942077,0.636544


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

Training Loss,Validation Loss,Epoch,F1
0.275966,0.942077,4,0.636544


fold 4 macro F1 (best checkpoint by eval macro F1): 0.6365

========== SUMMARY ==========
macro F1 per fold: [0.6332, 0.6884, 0.6214, 0.6648, 0.6365]
mean macro F1: 0.6489 (+/- 0.0244)


In [ ]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report, f1_score

df = pd.read_csv("../data/sentiment_sample_500_labelled.csv", encoding="utf-8-sig")
df["sentiment"] = df["sentiment"].str.strip().str.capitalize()
df = df[df["text"].notna() & (df["text"].astype(str).str.strip() != "")]

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual",
)

results = sentiment_pipe(df["text"].tolist(), truncation=True)

label_map = {"negative": "Negative", "neutral": "Neutral", "positive": "Positive"}
df["predicted"] = [label_map[r["label"].lower()] for r in results]

print(classification_report(df["sentiment"], df["predicted"], digits=3))
baseline_f1 = f1_score(df["sentiment"], df["predicted"], average="macro")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9647.15it/s]


              precision    recall  f1-score   support

    Negative      0.449     0.460     0.455        87
     Neutral      0.297     0.667     0.411       114
    Positive      0.807     0.412     0.545       294

    accuracy                          0.479       495
   macro avg      0.518     0.513     0.470       495
weighted avg      0.626     0.479     0.498       495

zero-shot macro F1: 0.4701


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

all_preds = []
all_labels = []

import glob

for fold_num, (train_ds, test_ds) in enumerate(folds):
    checkpoint_dirs = glob.glob(f"./results/fold_{fold_num}/checkpoint-*")
    if not checkpoint_dirs:
        raise FileNotFoundError(f"No checkpoint found for fold {fold_num}")
    checkpoint_path = checkpoint_dirs[0]

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
    tokenizer_fold = AutoTokenizer.from_pretrained(MODEL_NAME)

    test_ds_tok = test_ds.map(preprocess_fn, batched=True)
    test_ds_tok = test_ds_tok.remove_columns(["text", "sentiment"])
    test_ds_tok.set_format("torch")

    trainer = Trainer(model=model)
    preds_output = trainer.predict(test_ds_tok)
    preds = np.argmax(preds_output.predictions, axis=-1)
    labels = preds_output.label_ids

    fold_acc = accuracy_score(labels, preds)
    print(f"fold {fold_num} accuracy: {fold_acc:.4f}")

    all_preds.extend(preds)
    all_labels.extend(labels)

Map: 100%|██████████| 99/99 [00:00<00:00, 6803.69 examples/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


fold 0 accuracy: 0.7172


Map: 100%|██████████| 99/99 [00:00<00:00, 5269.36 examples/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


fold 1 accuracy: 0.7172


Map: 100%|██████████| 99/99 [00:00<00:00, 13005.80 examples/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


fold 2 accuracy: 0.6869


Map: 100%|██████████| 99/99 [00:00<00:00, 13155.37 examples/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


fold 3 accuracy: 0.7273


Map: 100%|██████████| 99/99 [00:00<00:00, 12567.30 examples/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


fold 4 accuracy: 0.6768

========== OUT-OF-FOLD CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

    Negative      0.630     0.667     0.648        87
     Neutral      0.500     0.491     0.496       114
    Positive      0.808     0.799     0.803       294

    accuracy                          0.705       495
   macro avg      0.646     0.652     0.649       495
weighted avg      0.706     0.705     0.705       495



In [ ]:
from datasets import Dataset

full_ds = Dataset.from_pandas(df[["text", "sentiment"]].reset_index(drop=True))
full_ds_tok = full_ds.map(preprocess_fn, batched=True)
full_ds_tok = full_ds_tok.remove_columns(["text", "sentiment"])
full_ds_tok.set_format("torch")

class_counts = df["sentiment"].value_counts()
class_weights = torch.tensor(
    [1.0 / class_counts[label] for label in label_list], dtype=torch.float
)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    id2label=id2label,
    label2id=label2id,
)

final_training_args = TrainingArguments(
    output_dir="./results/final_model",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=4,      
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    seed=RANDOM_STATE,
)

final_trainer = WeightedTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_ds_tok,
    class_weights=class_weights,
)

final_trainer.train()
final_trainer.save_model("./results/final_model")
tokenizer.save_pretrained("./results/final_model")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30579.82it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.916153
20,0.950664
30,0.935277
40,0.742041
50,0.629257
60,0.690803
70,0.540723
80,0.506153
90,0.509260
100,0.482091


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]
c:\Users\Keerthi\Documents\IDL Match Predictions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

saved final model to ./results/final_model


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./results/final_model"

df = pd.read_csv("../data/yt_routine_live_chat.csv", encoding="utf-8-sig")

df = df[df["text"].notna()]
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip() != ""]
df = df.reset_index(drop=True)

print(f"rows after cleaning: {len(df)}")
print(f"any NaN left: {df['text'].isna().sum()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

id2label = model.config.id2label

texts = [str(t) for t in df["text"].tolist()]
batch_size = 32
predictions = []

with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().tolist()
        predictions.extend(preds)

df["sentiment"] = [id2label[p] for p in predictions]

df.to_csv("../data/yt_live_chat_sentiment.csv", index=False, encoding="utf-8-sig")

rows after cleaning: 3896
any NaN left: 0


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12353.18it/s]


saved 3896 rows with sentiment predictions to ../data/yt_live_chat_sentiment.csv
